# 04 — Synthesize + utilise the graph

End-to-end retrieval + synthesis over the loaded graph (notebook 03). Demonstrates:

1. **EN query** → jurisdiction inferred (EU), language detected (`en`), synthesis with common-law system prompt.
2. **DA query** → jurisdiction inferred (DK), language detected (`da`), synthesis with civil-law system prompt + DA disclaimer (Phase 8).
3. **Cross-jurisdiction query** → DK + EU candidates returned, IMPLEMENTS edge surfaces.
4. **Caller-knows-best override** → `--jurisdiction X` works even when X is disabled (Phase 11 invariant).
5. **Audit metadata** → `Answer.to_dict()` carries resolved `language`, `jurisdiction`, `as_of`, `citations`, `caveats`.

Uses `FakeSynthesizer` for deterministic offline demo — swap in `get_synthesizer('anthropic')` for real Claude output (needs `ANTHROPIC_API_KEY`).

## Prereqs
- Notebook 03 ran (graph populated with EU GDPR + DCD + DK lbk + IMPLEMENTS + chunks).

In [ ]:
import os
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

from crimellm.clg.config import get_settings
from crimellm.clg.embed.embedder import FakeEmbedder
from crimellm.clg.graph import get_store
from crimellm.clg.retrieval.parse_query import parse_query
from crimellm.clg.retrieval.seed import seed_from_chunks
from crimellm.clg.retrieval.synthesize import FakeSynthesizer, get_synthesizer

get_settings.cache_clear()
store = get_store()
store.verify()

# Match the embedder used in notebook 03.
embedder = FakeEmbedder(dim=store.settings.embedding_dim)

# Switch to a real Claude synthesizer when ANTHROPIC_API_KEY is set:
# synth = get_synthesizer('anthropic')
synth = FakeSynthesizer()

print('store      :', store.settings.neo4j_uri)
print('embedder   :', embedder.name, 'dim', embedder.dim)
print('synthesizer:', synth.name)

## Section 1 — parse_query shows the heuristics in action

`parse_query` infers `jurisdiction` from cue tables (Phase 4.5 4-way argmax) and `language` from the multi-signal detector (Phase 7 + 14.3 — DA/EN/FR/DE).

In [ ]:
samples = [
    'What does Article 6 GDPR say about lawful processing?',
    'Hvilke regler gælder for behandling af persondata efter loven?',
    'Har Højesteret fraveget U.2010.456H?',
    'Quelles sont les obligations sous la directive 2019/770?',
    'Hat der Gerichtshof Artikel 101 AEUV ausgelegt?',
    'a generic question with no jurisdiction cue',
]
for s in samples:
    q = parse_query(s)
    print(f"{q.language}({q.language_confidence:.2f}) {str(q.jurisdiction):<5} | {s}")

## Section 2 — EN query → EU subgraph → EN synthesis

The English GDPR question routes to EU and gets the English system prompt + English disclaimer (Phase 8).

In [ ]:
q = parse_query('What does Article 6 GDPR say about lawful processing?')
print('jurisdiction:', q.jurisdiction, '/ language:', q.language)

candidates = seed_from_chunks(q, embedder, k=3, store=store)
ans = synth.synthesise(query=q, candidates=candidates, good_law={})
print()
print(ans.text)

## Section 3 — DA query → DK subgraph → DA synthesis

Detector picks `da`; `parse_query` infers DK via cue table; synth emits DA disclaimer + DA body prose.

In [ ]:
q = parse_query(
    'Hvilke regler gælder for behandling af persondata efter databeskyttelsesloven?'
).with_overrides(jurisdiction='DK')   # caller-knows-best — bypasses any auto-route

print('jurisdiction:', q.jurisdiction, '/ language:', q.language)
candidates = seed_from_chunks(q, embedder, k=3, store=store)
print()
for c in candidates:
    print(f'  - {c.parent_jurisdiction} {c.parent_id}  score={c.base_score:.3f}')

ans = synth.synthesise(query=q, candidates=candidates, good_law={})
print()
print(ans.text)

## Section 4 — Cross-jurisdiction: IMPLEMENTS edge traversal

A query that crosses jurisdictions naturally pulls in both endpoints. Here we ask Cypher directly to surface the IMPLEMENTS edge our DK lbk + EU regulation form.

In [ ]:
rows = store.run(
    'MATCH (dk:Instrument {jurisdiction:"DK"})-[:IMPLEMENTS]->(eu:Instrument {jurisdiction:"EU"}) '
    'RETURN dk.id AS dk, dk.short_title AS dk_title, eu.id AS eu, eu.short_title AS eu_title'
)
for r in rows:
    print(f"{r['dk']:<32} \u2192 {r['eu']}")
    print(f"  '{r['dk_title']}'")
    print(f"  IMPLEMENTS '{r['eu_title'][:60]}…'")
    print()

## Section 5 — Removability + caller-knows-best (Phase 11)

Flip `ENABLED_JURISDICTIONS=EU` → DK is silenced in retrieval. But `--jurisdiction DK` override still surfaces DK data on demand. Data is never deleted.

In [ ]:
os.environ['ENABLED_JURISDICTIONS'] = 'EU'
get_settings.cache_clear()

# 1) Default search: DK silenced, only EU surfaces.
q = parse_query('Behandling af persondata')
candidates = seed_from_chunks(q, embedder, k=5, store=store)
print('with DK disabled, juris in hits:', sorted({c.parent_jurisdiction for c in candidates}))

# 2) Override: --jurisdiction DK still works.
q_override = q.with_overrides(jurisdiction='DK')
candidates_override = seed_from_chunks(q_override, embedder, k=5, store=store)
print('with --jurisdiction DK override, juris in hits:',
      sorted({c.parent_jurisdiction for c in candidates_override}))

# Restore the full enabled set for downstream cells.
os.environ['ENABLED_JURISDICTIONS'] = 'US,EW,UK,EU,DK'
get_settings.cache_clear()

## Section 6 — Answer.to_dict carries audit metadata

Every synthesised answer surfaces the resolved query state (`jurisdiction`, `language`, `as_of`) + citation guard outcome (`citations` valid, `caveats` fabricated/good-law).

In [ ]:
q = parse_query('Hvad regulerer denne lov?').with_overrides(jurisdiction='DK')
ans = synth.synthesise(
    query=q,
    candidates=seed_from_chunks(q, embedder, k=3, store=store),
    good_law={},
)
d = ans.to_dict()
print(json.dumps({k: v for k, v in d.items() if k not in {'used'}}, indent=2, ensure_ascii=False))

## Section 7 — CLI equivalent

Everything above is also available as `clg query`:

In [ ]:
import subprocess

r = subprocess.run(
    [
        'uv', 'run', 'clg', 'query',
        'Hvilke regler g\u00e6lder for behandling af persondata?',
        '--jurisdiction', 'DK',
        '--lang', 'da',
        '--synth', 'fake',
        '--json',
    ],
    capture_output=True, text=True,
)
print('exit:', r.returncode)
if r.returncode == 0:
    out = json.loads(r.stdout)
    print('text:'); print(out['text'])
    print('language    :', out['language'])
    print('jurisdiction:', out['jurisdiction'])
    print('citations   :', out['citations'])
else:
    print(r.stderr)

## Section 8 — Eval gold set

Phase 10 ships 24 hand-curated gold questions across US (2) + UK (4) + EU (5) + DK (10) + cross-jurisdiction (3). Filter at runtime:

In [ ]:
from crimellm.clg.eval.schema import load_gold_set

gold = load_gold_set('data/eval/seed.yaml')
print('total questions:', len(gold))
for j in ('US', 'UK', 'EU', 'DK', 'XJ'):
    sub = gold.filter_by_jurisdiction([j])
    print(f'  {j}: {len(sub):>2}')

# Run only DK + cross-jurisdiction via the CLI:
# uv run clg eval --gold-set data/eval/seed.yaml --jurisdiction DK,XJ --synth fake

## Where to go next

- **Real Claude synth:** export `ANTHROPIC_API_KEY` and swap `FakeSynthesizer()` for `get_synthesizer('anthropic')`.
- **Real embedder:** `get_embedder('sentence-transformers', model='BAAI/bge-m3')` (1024-d, same as the demo index) or `Qwen/Qwen3-Embedding-8B` (4096-d, needs `clg graph rebuild-vector-index --dim 4096`).
- **Treatment classification:** `clg link distill` → `clg link train-distilled` → `clg link treatment --backend rules+distilled+ollama` populates `treatment` on every CITES edge. Phase 6 cascade routes DA / EU rules per-jurisdiction.
- **Production deployment:** see `docs/jurisdiction-management.local.md` for the operator playbook (enable / disable / re-enable, OCR setup for older DK PDFs, embedder swaps).